### Import LLM-As-A-Judge Annotated Trace Data

In [1]:
from huggingface_hub import hf_hub_download
import pandas as pd
import json

REPO_ID = "mcemri/MAD"
FILENAME = "MAD_full_dataset.json"

file_path =  hf_hub_download(repo_id=REPO_ID, filename=FILENAME, repo_type="dataset")
with open(file_path, "r") as f:
    llm_data = json.load(f)

print(f"Loaded {len(llm_data)} records (full dataset).")

/Users/indigosahil/Desktop/agentic-reliability-research/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 1242 records (full dataset).


### Import Human-Annotated Trace Data

In [2]:
FILENAME = "MAD_human_labelled_dataset.json"

file_path =  hf_hub_download(repo_id=REPO_ID, filename=FILENAME, repo_type="dataset")
with open(file_path, "r") as f:
    data = json.load(f)

print(f"Loaded {len(data)} records (human labelled).")

Loaded 19 records (human labelled).


### Schema — full (LLM-judge) dataset

Each record: `mas_name`, `llm_name`, `benchmark_name`, `trace_id`, `trace` (`key`/`index`/`trajectory` — raw text log, can be 100K+ chars), `mast_annotation` (14 binary flags, codes `1.1`–`3.3`).

This is **multi-label binary classification over a whole trajectory** — not step-level localization. Traces come from *other* MAS frameworks (AG2, ChatDev, MetaGPT, Magentic, OpenManus, AppWorld, HyperAgent), not from our own LangGraph/LangSmith runs — there's no live coupling, this is purely offline detector training data.

In [3]:
import collections

print("=== benchmark_name distribution ===")
print(collections.Counter(d["benchmark_name"] for d in llm_data))
print()
print("=== mas_name distribution ===")
print(collections.Counter(d["mas_name"] for d in llm_data))
print()
print("=== llm_name distribution ===")
print(collections.Counter(d["llm_name"] for d in llm_data))
print()

codes = list(llm_data[0]["mast_annotation"].keys())
any_fail = sum(1 for d in llm_data if any(v == 1 for v in d["mast_annotation"].values()))
print(f"records with >=1 failure mode flagged: {any_fail} / {len(llm_data)}")

per_code = collections.Counter()
for d in llm_data:
    for k, v in d["mast_annotation"].items():
        if v == 1:
            per_code[k] += 1
print("per-code positive counts:", dict(sorted(per_code.items())))

=== benchmark_name distribution ===
Counter({'ProgramDev': 390, 'GSM': 223, 'Olympiad': 206, 'GAIA': 195, 'MMLU': 168, 'Test-C': 30, 'SWE-Bench-Lite': 30})

=== mas_name distribution ===
Counter({'AG2': 597, 'MetaGPT': 230, 'Magentic': 195, 'ChatDev': 130, 'OpenManus': 30, 'AppWorld': 30, 'HyperAgent': 30})

=== llm_name distribution ===
Counter({'GPT-4o': 919, 'Claude': 323})

records with >=1 failure mode flagged: 956 / 1242
per-code positive counts: {'1.1': 367, '1.2': 10, '1.3': 451, '1.4': 71, '1.5': 346, '2.1': 50, '2.2': 303, '2.3': 224, '2.4': 40, '2.6': 495, '3.1': 208, '3.2': 216, '3.3': 283}


### Schema - human-labelled dataset (19 records)

Different shape from the full dataset: `round`, `mas_name`, `benchmark_name`, `trace_id`, `trace`, `annotations`.
`annotations` is a **list of 16 failure-mode dicts** (not 14 — adds `3.4 Waiting for known information`, `4.1`, `4.2`, `4.3`; drops the `1.x`–`3.x`-only framing of the full set), each with `annotator_1/2/3: bool` + a `failure mode` field carrying the **code, name, and full definition text inline** — useful as a free taxonomy reference (printed below) without needing the original paper.

This is the inter-annotator-agreement subset, not extra training volume — useful for checking annotation reliability per failure mode, not for boosting label count.

In [4]:
print("count:", len(data))
print("keys:", list(data[0].keys()))
print()

# taxonomy reference — code + name + definition, pulled straight from the "failure mode" field
for ann in data[0]["annotations"]:
    print(ann["failure mode"].split("\n")[0])

print()
# inter-annotator agreement: how often do all 3 annotators agree, per failure mode?
agree_counts = collections.Counter()
total_counts = collections.Counter()
for record in data:
    for ann in record["annotations"]:
        code = ann["failure mode"].split("\n")[0]
        votes = [ann["annotator_1"], ann["annotator_2"], ann["annotator_3"]]
        total_counts[code] += 1
        if len(set(votes)) == 1:
            agree_counts[code] += 1

print("full 3/3 agreement rate per failure mode:")
for code in total_counts:
    print(f"  {code[:40]:40s} {agree_counts[code]}/{total_counts[code]}")

count: 19
keys: ['round', 'mas_name', 'benchmark_name', 'trace_id', 'trace', 'annotations']

1.1 Poor task constraint compliance
1.2 Inconsistency between reasoning and action
1.3 Undetected conversation ambiguities and contradictions
1.4 Fail to elicit clarification
1.5 Unaware of stopping conditions
2.1 Unbatched repetitive execution
2.2 Step repetition
2.3 Backtracking interruption
2.4 Conversation reset 
2.5 Derailment from task
2.6 Disobey role specification
3.1 Disagreement induced inaction
3.2 Withholding relevant information
3.3 Ignoring suggestions from agents
3.4 Waiting for known information
4.1 Ill specified termination condition leading to premature termination
4.3 Lack of critical verification
4.2 Lack of result verification

full 3/3 agreement rate per failure mode:
  1.1 Poor task constraint compliance      15/15
  1.2 Inconsistency between reasoning and  15/15
  1.3 Undetected conversation ambiguities  5/5
  1.4 Fail to elicit clarification         5/5
  1.5 Unaware of